# Retail banking capacity simulation

**Independent Python extension of an Operations Analytics report.** Synthetic assumptions only. Run all cells to reproduce the results. The original written report and this implementation are distinct pieces of work.

## Assumptions

- 8.5 arrivals/hour; exponential gaps.
- Lognormal service times: mean 12 min, CV 0.25.
- 6-hour opening day; 30 days per replication, 30 replications.
- Compare 2 and 3 officers with the **same synthetic customers**.
- First-come, first-served; all customers arriving before closing finish service.
- SLA: service begins within 15 minutes.
- Labour: LKR 2,000/officer-hour.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Works when the notebook is in analysis/ or opened from the project root.
script_dir = Path.cwd()
if not (script_dir / "simulation.py").exists():
    script_dir = script_dir / "analysis"
sys.path.insert(0, str(script_dir.resolve()))
from simulation import run_experiment

results, customers = run_experiment()
summary = results.groupby("officers").mean(numeric_only=True).drop(columns="replication")
summary.round(2)

## Compare staffing

The output is an estimate from 30 seeded replications. The extra officer should reduce average and extreme waits, with higher scheduled cost. This is **not** a claim about a real bank.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot = summary[["mean_wait_min", "p95_wait_min"]].rename(columns={"mean_wait_min":"Average wait", "p95_wait_min":"95th-percentile wait"})
plot.plot(kind="bar", ax=ax)
ax.set(xlabel="Officers", ylabel="Minutes", title="Waiting times by staffing level")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
summary["sla_within_15_pct"].plot(kind="bar", ax=ax)
ax.set(xlabel="Officers", ylabel="Percent", title="Service starts within 15 minutes", ylim=(0, 105))
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## Interpretation and limitations

The third officer reduces queueing risk but increases scheduled labour cost by 50%. Each day starts with an empty queue, arrivals stop after six hours, and existing customers are served to completion. The utilisation metric counts busy minutes **inside** operating hours. We do not price work after closing, model customer abandonment, or claim these results replicate the original assignment exactly.